# Exportable a Ollama (GGUF): Causal Fine-Tuning
Este notebook cambia la forma del entrenamiento para **garantizar compatibilidad nativa con Ollama y GGUF**.

En lugar de sustituir el Tokenizador numéricamente, convertiremos las estadísticas del csv en **Prompts de Texto Instructivo**.

## 1. Dependencias y llama.cpp

In [ ]:
!pip install -q kagglehub transformers datasets peft accelerate trl
!git clone https://github.com/ggerganov/llama.cpp.git || true
!pip install -q -r llama.cpp/requirements.txt

import pandas as pd
import torch
import os
import glob
from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer
import kagglehub

## 2. Preparar Datos (Formateo Texto a Prompt)

In [ ]:
MUESTRAS_BENIGN = 1000
MUESTRAS_ATAQUE = 500
path = kagglehub.dataset_download("mkashifn/nbaiot-dataset")

def cargar_a_texto(patron, clase):
    archivos = glob.glob(os.path.join(path, '**', patron), recursive=True)
    if not archivos: archivos = glob.glob(os.path.join(path, patron))
    prompts = []
    for f in archivos:
        try:
            df = pd.read_csv(f, nrows=500)
            for _, row in df.iterrows():
                texto = f"Red: MI_dir_L5: {row.get('MI_dir_L5_weight',0):.2f}, H_L5: {row.get('H_L5_weight',0):.2f}"
                prompt = f"<|user|>\n{texto}\n<|assistant|>\nTrafico: {clase}<|endoftext|>"
                prompts.append({'text': prompt})
        except e: pass
        if len(prompts)>500: break
    return prompts

datos = cargar_a_texto('*.benign.csv', 'Normal') + cargar_a_texto('*.mirai.*.csv', 'Mirai')
hf_dataset = Dataset.from_list(datos).shuffle(seed=42)
print(hf_dataset[0]['text'])

## 3. Entrenamiento (Causal LM y SFTTrainer)

In [ ]:
modelo_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(modelo_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(modelo_id, device_map="auto", torch_dtype=torch.float16)
peft = LoraConfig(r=8, target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM")

trainer = SFTTrainer(
    model=model, train_dataset=hf_dataset, dataset_text_field="text",
    max_seq_length=128, peft_config=peft,
    args=TrainingArguments(output_dir="./lora_out", per_device_train_batch_size=8, num_train_epochs=1)
)
trainer.train()
trainer.model.save_pretrained("./mi_lora_causal")

## 4. Fusión (Merge) y Conversión GGUF

In [ ]:
import gc
del model, trainer; gc.collect(); torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(modelo_id, torch_dtype=torch.float16, device_map="cpu")
modelo_f = PeftModel.from_pretrained(base_model, "./mi_lora_causal").merge_and_unload()
modelo_f.save_pretrained("../modelos_entrenados/Botnet_HF")
tokenizer.save_pretrained("../modelos_entrenados/Botnet_HF")

print("Convirtiendo a GGUF...")
!python llama.cpp/convert_hf_to_gguf.py ../modelos_entrenados/Botnet_HF --outfile ../modelos_entrenados/botnet.gguf --outtype q8_0
print("\n\U0001f389 LISTO. Tu modelo compatible con Ollama está en: ../modelos_entrenados/botnet.gguf")